In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import log_loss, brier_score_loss

project_root = Path.cwd().parents[2]

feature_dir = (
    project_root / "data" / "processed" / "engineered_features"
    / "group_3_other_skill_history" / "v3.4"
)

prediction_dir = (
    project_root / "data" / "processed" / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

print("Feature directory:", feature_dir)
print("Prediction directory:", prediction_dir)

Feature directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_3_other_skill_history\v3.4
Prediction directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions


In [2]:
v34_fold_datasets = []

for fold_number in range(1, 6):

    train_path = feature_dir / f"fold_{fold_number}_train.csv"
    val_path = feature_dir / f"fold_{fold_number}_validation.csv"

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    v34_fold_datasets.append((train_df, val_df))

    print(
        f"Fold {fold_number}: "
        f"Train = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

print("\nAll V3.4 datasets loaded successfully!")

Fold 1: Train = (177324, 5), Validation = (42659, 5)
Fold 2: Train = (176273, 5), Validation = (43710, 5)
Fold 3: Train = (170186, 5), Validation = (49797, 5)
Fold 4: Train = (178311, 5), Validation = (41672, 5)
Fold 5: Train = (177838, 5), Validation = (42145, 5)

All V3.4 datasets loaded successfully!


In [3]:
feature_columns = [
    "weighted_other_skill_smoothed_accuracy",
    "log_weighted_other_skill_previous_interactions"
]

target_column = "correct"

for fold_number, (train_df, val_df) in enumerate(
    v34_fold_datasets, start=1
):
    print(f"\n========== Fold {fold_number} ==========")

    print("Training missing values:", train_df.isna().sum().sum())
    print("Validation missing values:", val_df.isna().sum().sum())

    print("\nTraining feature ranges:")
    print(train_df[feature_columns].agg(["min", "max"]))

    print("\nValidation feature ranges:")
    print(val_df[feature_columns].agg(["min", "max"]))

    print("\nTraining target distribution:")
    print(train_df[target_column].value_counts(normalize=True))

    print("\nValidation target distribution:")
    print(val_df[target_column].value_counts(normalize=True))


========== Fold 1 ==========
Training missing values: 0
Validation missing values: 0

Training feature ranges:
     weighted_other_skill_smoothed_accuracy  \
min                                0.033701   
max                                1.000000   

     log_weighted_other_skill_previous_interactions  
min                                        0.000000  
max                                        6.766192  

Validation feature ranges:
     weighted_other_skill_smoothed_accuracy  \
min                                0.061581   
max                                0.935096   

     log_weighted_other_skill_previous_interactions  
min                                        0.000000  
max                                        6.859615  

Training target distribution:
correct
1    0.660136
0    0.339864
Name: proportion, dtype: float64

Validation target distribution:
correct
1    0.652336
0    0.347664
Name: proportion, dtype: float64

========== Fold 2 ==========
Training missing val

In [4]:
feature_columns = [
    "weighted_other_skill_smoothed_accuracy",
    "log_weighted_other_skill_previous_interactions"
]

target_column = "correct"

v34_models = []
v34_fold_results = []
v34_predictions = []

for fold_number, (train_df, val_df) in enumerate(
    v34_fold_datasets, start=1
):
    print(f"\nTraining Fold {fold_number}...")

    # Separate training features and target
    X_train = train_df[feature_columns]
    y_train = train_df[target_column]

    # Separate validation features and actual target
    X_val = val_df[feature_columns]
    y_val = val_df[target_column]

    # Create Logistic Regression pipeline
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ))
    ])

    # Train model
    model.fit(X_train, y_train)

    # Predict probability of correct outcome (class 1)
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Calculate fold-level metrics
    fold_log_loss = log_loss(y_val, y_pred_proba)
    fold_brier = brier_score_loss(y_val, y_pred_proba)

    # Store trained model
    v34_models.append(model)

    v34_fold_results.append({
        "fold": fold_number,
        "log_loss": fold_log_loss,
        "brier_score": fold_brier
    })

    # Store actual labels and predicted probabilities
    fold_predictions = pd.DataFrame({
        "user_id": val_df["user_id"].values,
        "order_id": val_df["order_id"].values,
        "fold": fold_number,
        "y_true": y_val.values,
        "y_pred_proba": y_pred_proba,
        "model": "V3.4"
    })

    v34_predictions.append(fold_predictions)

    print(f"Log Loss: {fold_log_loss:.4f}")
    print(f"Brier Score: {fold_brier:.4f}")
    print(f"Predictions generated: {len(fold_predictions)}")


# Combine predictions from all five validation folds
v34_predictions_df = pd.concat(
    v34_predictions,
    ignore_index=True
)

# Save OOF predictions
prediction_path = prediction_dir / "v3_4_predictions.csv"

v34_predictions_df.to_csv(
    prediction_path,
    index=False
)

print("\n========== V3.4 COMPLETED ==========")

print(
    "Average Log Loss:",
    np.mean([r["log_loss"] for r in v34_fold_results])
)

print(
    "Average Brier Score:",
    np.mean([r["brier_score"] for r in v34_fold_results])
)

print("\nTotal OOF predictions:", len(v34_predictions_df))
print("Prediction file:", prediction_path)

print("\nAll five V3.4 models trained and predictions saved!")


Training Fold 1...
Log Loss: 0.6161
Brier Score: 0.2134
Predictions generated: 42659

Training Fold 2...
Log Loss: 0.6112
Brier Score: 0.2109
Predictions generated: 43710

Training Fold 3...
Log Loss: 0.6291
Brier Score: 0.2192
Predictions generated: 49797

Training Fold 4...
Log Loss: 0.6174
Brier Score: 0.2138
Predictions generated: 41672

Training Fold 5...
Log Loss: 0.6092
Brier Score: 0.2101
Predictions generated: 42145

========== V3.4 COMPLETED ==========
Average Log Loss: 0.6165931514921624
Average Brier Score: 0.21346496224248881

Total OOF predictions: 219983
Prediction file: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v3_4_predictions.csv

All five V3.4 models trained and predictions saved!


In [5]:
prediction_files = {
    "V3.1": "v3_1_predictions.csv",
    "V3.2": "v3_2_predictions.csv",
    "V3.3": "v3_3_predictions.csv",
    "V3.4": "v3_4_predictions.csv"
}

for model_name, filename in prediction_files.items():

    df = pd.read_csv(prediction_dir / filename)

    print(f"\n========== {model_name} ==========")
    print("Shape:", df.shape)
    print("Missing values:", df.isna().sum().sum())
    print(
        "Duplicate interactions:",
        df.duplicated(["user_id", "order_id"]).sum()
    )
    print("Probability range:",
          df["y_pred_proba"].min(),
          df["y_pred_proba"].max())


========== V3.1 ==========
Shape: (219983, 6)
Missing values: 0
Duplicate interactions: 0
Probability range: 0.166564768173653 0.8647148598987755

========== V3.2 ==========
Shape: (219983, 6)
Missing values: 0
Duplicate interactions: 0
Probability range: 0.1616308329644777 0.8674175514155165

========== V3.3 ==========
Shape: (219983, 6)
Missing values: 0
Duplicate interactions: 0
Probability range: 0.1633874840488959 0.8682982207940267

========== V3.4 ==========
Shape: (219983, 6)
Missing values: 0
Duplicate interactions: 0
Probability range: 0.1577610595054825 0.8717542197435731


In [6]:
prediction_dfs = {}

for model_name, filename in prediction_files.items():
    prediction_dfs[model_name] = pd.read_csv(
        prediction_dir / filename
    ).sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)


reference = prediction_dfs["V3.1"]

for model_name, df in prediction_dfs.items():

    same_interactions = (
        reference[["user_id", "order_id"]].equals(
            df[["user_id", "order_id"]]
        )
    )

    same_labels = reference["y_true"].equals(df["y_true"])

    print(f"\n{model_name}")
    print("Same interactions:", same_interactions)
    print("Same actual labels:", same_labels)


V3.1
Same interactions: True
Same actual labels: True

V3.2
Same interactions: True
Same actual labels: True

V3.3
Same interactions: True
Same actual labels: True

V3.4
Same interactions: True
Same actual labels: True
